# TA-BN-ODE-DSTPP: Full Reproducibility Notebook

**Paper:** *Temporal Adaptive Neural Ordinary Differential Equations with Deep Spatio-Temporal Point Processes for Real-Time Network Intrusion Detection*

**Authors:** Roger Nick Anaedevha, Alexander Gennadevich Trofimov, Yuri Vladimirovich Borodachev

**Journal:** Complex and Intelligent Systems (Q1)

**Datasets:**
- ICS3D: DOI [10.34740/kaggle/dsv/12483891](https://doi.org/10.34740/kaggle/dsv/12483891)
- Benchmarks: DOI [10.34740/KAGGLE/DSV/12479689](https://doi.org/10.34740/KAGGLE/DSV/12479689)

---

This notebook reproduces the main experimental results from the paper,
including training, evaluation, ablation study, calibration, and online
adaptation with concept drift detection.

## 0. Setup and Installation

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch>=2.0 torchdiffeq numpy pandas scikit-learn scipy
# !pip install matplotlib seaborn tqdm kagglehub pyro-ppl transformers accelerate

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

from configs.default import Config
from models.full_model import TABNODEPointProcess
from models.bayesian import BayesianWrapper
from data.loader import ICS3DLoader, BenchmarkLoader
from data.preprocessing import preprocess_dataset, temporal_split, TimeSeriesDataset
from utils.training import Trainer
from utils.evaluation import Evaluator, compute_ece, compute_psi
from utils.online import OnlineAdapter

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = Config()
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Load and Preprocess Data

We load the ICS3D and benchmark datasets. Datasets are automatically
downloaded from Kaggle. Ensure `KAGGLE_USERNAME` and `KAGGLE_KEY`
environment variables are set, or place `kaggle.json` in `~/.kaggle/`.

In [ ]:
# Load Container Security dataset (ICS3D)
ics3d = ICS3DLoader(cfg.data.ics3d_kaggle_slug)
df_container, _ = ics3d.load_container_security()
print(f'Container Security: {len(df_container):,} records, {df_container.shape[1]} columns')
df_container.head()

In [ ]:
# Preprocess
X, y, label_encoder, scaler = preprocess_dataset(df_container)

# Temporal split: 70/15/15 (Section 5.1)
X_train, y_train, X_val, y_val, X_test, y_test = temporal_split(
    X, y, cfg.data.temporal_split_ratios
)

print(f'\nTrain: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Features: {X.shape[1]} | Classes: {len(label_encoder.classes_)}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Create PyTorch datasets
train_ds = TimeSeriesDataset(X_train, y_train)
val_ds = TimeSeriesDataset(X_val, y_val)
test_ds = TimeSeriesDataset(X_test, y_test)

print(f'Dataset sizes — Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 2. Model Architecture

The TA-BN-ODE-DSTPP model integrates:
- **Temporal Adaptive Batch Normalization Neural ODEs** (Eq. 1, 4, 5, 7)
- **Deep Spatio-Temporal Point Processes** with transformer intensity (Eq. 8, 11)
- **Structured Variational Bayesian Inference** (Eq. 12)

Architecture: hidden_dim=256, model_dim=512, 2 ODE blocks, 4 time constants,
4 transformer layers, 8 attention heads. Total: ~2.3M parameters.

In [ ]:
input_dim = X.shape[1]
n_classes = len(label_encoder.classes_)

model = TABNODEPointProcess(
    input_dim=input_dim,
    hidden_dim=cfg.model.hidden_dim,
    n_classes=n_classes,
    d_model=cfg.model.model_dim,
    n_ode_blocks=cfg.model.n_ode_blocks,
    time_constants=cfg.model.time_constants,
    n_transformer_layers=cfg.model.n_transformer_layers,
    n_attention_heads=cfg.model.n_attention_heads,
    tabn_mlp_hidden=cfg.model.tabn_mlp_hidden,
    tabn_mlp_layers=cfg.model.tabn_mlp_layers,
    solver_method=cfg.model.solver_method,
    rtol=cfg.model.solver_rtol,
    atol=cfg.model.solver_atol,
    transformer_dropout=cfg.model.transformer_dropout,
)

n_params = sum(p.numel() for p in model.parameters())
size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)
print(f'Model parameters: {n_params:,} ({size_mb:.1f} MB)')
print(f'Target from paper: ~2.3M parameters, ~9.2MB')

## 3. Training

Training setup (Section 5.1):
- Adam optimizer, lr=1e-3 with cosine annealing to 1e-5
- Batch size 256, max 100 epochs with early stopping (patience=10)
- Gradient clipping: max norm 1.0
- Bayesian: 10 MC samples during training

In [ ]:
# Optional: Bayesian wrapper for structured variational inference
bayesian = BayesianWrapper(model, rank=cfg.model.low_rank_dim)

trainer = Trainer(
    model, device,
    lr=cfg.training.learning_rate,
    min_lr=cfg.training.min_learning_rate,
    batch_size=cfg.training.batch_size,
    max_epochs=cfg.training.max_epochs,
    patience=cfg.training.early_stopping_patience,
    grad_clip=cfg.training.grad_clip_norm,
    loss_weights={
        'cls': cfg.training.weight_cls,
        'tpp': cfg.training.weight_tpp,
        'reg': cfg.training.weight_reg,
    },
    bayesian_wrapper=bayesian,
)

history = trainer.train(train_ds, val_ds)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.get('train_total', history.get('train_elbo', [])))
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True)

axes[1].plot(history.get('val_accuracy', []))
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy'); axes[1].grid(True)

axes[2].plot(history.get('lr', []))
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate (Cosine Annealing)'); axes[2].grid(True)

plt.tight_layout()
plt.savefig('../outputs/training_curves.png', dpi=150)
plt.show()

## 4. Temperature Calibration

Post-hoc temperature scaling on validation data (Section 4.4).
Target ECE: 0.017

In [ ]:
val_loader = DataLoader(val_ds, batch_size=cfg.training.eval_batch_size, shuffle=False)
model.calibrate_temperature(val_loader, device)

## 5. Evaluation

Comprehensive evaluation: accuracy, F1, AUC, ECE, throughput.
Uses 50 MC samples at test time for uncertainty quantification.

In [ ]:
evaluator = Evaluator(model, device)
test_loader = DataLoader(test_ds, batch_size=cfg.training.eval_batch_size, shuffle=False)

results = evaluator.evaluate(
    test_loader,
    label_names=list(label_encoder.classes_),
    n_mc_samples=cfg.model.mc_samples_test,
    bayesian_wrapper=bayesian,
)

In [ ]:
# Model size and throughput
evaluator.model_size()
throughput = evaluator.measure_throughput(input_dim, cfg.training.batch_size)

## 6. Online Adaptation with Concept Drift Detection

Algorithm S2: EWC + DP-SGD online adaptation triggered by PSI-based
concept drift detection (PSI > 0.2 threshold).

In [ ]:
adapter = OnlineAdapter(
    model, device,
    ewc_lambda=cfg.online.ewc_lambda,
    ema_rho=cfg.online.ema_rho,
    base_lr=cfg.online.online_lr,
    mini_epochs=cfg.online.online_mini_epochs,
    psi_threshold=cfg.online.psi_threshold,
)

# Set reference distribution from training data
model.eval()
t_span = torch.linspace(0, 1, 10).to(device)
ref_confs = []
with torch.no_grad():
    for batch in DataLoader(train_ds, batch_size=256, shuffle=False):
        out = model(batch['x'].to(device), t_span)
        probs = torch.softmax(out['logits'], dim=1)
        ref_confs.extend(probs.max(dim=1)[0].cpu().numpy())
adapter.drift_detector.set_reference(np.array(ref_confs))

# Simulate streaming
stream_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
correct = 0; total = 0
for batch in stream_loader:
    result = adapter.process_batch(batch['x'], batch['y'])
    correct += (result['predictions'] == batch['y']).sum().item()
    total += len(batch['y'])

print(f'\nAdaptive streaming accuracy: {correct/total:.4f}')

## 7. Save Results

In [ ]:
import json

os.makedirs('../outputs', exist_ok=True)

# Save model
torch.save(model.state_dict(), '../outputs/ta_bn_ode_dstpp.pt')

# Save results
with open('../outputs/results.json', 'w') as f:
    json.dump({k: float(v) if isinstance(v, (float, np.floating)) else v
               for k, v in results.items()}, f, indent=2)

print('Model and results saved to ../outputs/')
print('\nDone! All results reproduced.')